# catalog-client quickstart

This notebook walks through the core workflows of the `catalog-client` library:

1. Connecting to the catalog
2. Registering a dataset
3. Listing and retrieving datasets
4. Creating and managing collections
5. Recording lineage edges
6. Error handling
7. Async usage

> **Prerequisites:** `pip install catalog-client`

In [ ]:
# Configuration — fill these in before running
BASE_URL = "https://your-catalog.example.com"
API_TOKEN = "your-api-token-here"

## 1. Connect

In [ ]:
from catalog_client import CatalogClient

# The context manager ensures the HTTP connection is closed on exit.
# For exploratory work you can also call client.close() manually.
client = CatalogClient(base_url=BASE_URL, api_token=API_TOKEN)

## 2. Register a dataset (fluent builder)

The registration builder is the recommended way to create a dataset.
It enforces that all required fields are provided before submitting.

In [ ]:
from catalog_client import (
    AssetType,
    DatasetModality,
    LineageType,
    OntologyEntry,
)

with CatalogClient(base_url=BASE_URL, api_token=API_TOKEN) as client:
    raw_dataset_id = (
        client.new_registration(
            canonical_id="pbmc-rnaseq-batch42",
            name="PBMC RNA-seq batch 42",
            version="1.0.0",
            project="atlas",
            modality=DatasetModality.sequencing,
        )
        .described("Bulk RNA-seq from 10 PBMC donors, batch 42.")
        .with_location(
            "s3://my-bucket/rnaseq/batch42/raw/",
            asset_type=AssetType.folder,
            file_format="FASTQ",
        )
        .with_governance(
            data_owner="genomics-team",
            data_sensitivity="internal",
            is_pii=False,
            is_phi=False,
        )
        .with_sample(
            organism=[
                OntologyEntry(label="Homo sapiens", ontology_id="NCBITaxon:9606")
            ]
        )
        .with_experiment(sub_modality="bulk")
        .submit()
    )

print("Registered raw dataset:", raw_dataset_id)

### Register a derived dataset with lineage

In [ ]:
with CatalogClient(base_url=BASE_URL, api_token=API_TOKEN) as client:
    processed_dataset_id = (
        client.new_registration(
            canonical_id="pbmc-rnaseq-batch42-processed",
            name="PBMC RNA-seq batch 42 (processed)",
            version="1.0.0",
            project="atlas",
            modality=DatasetModality.sequencing,
        )
        .with_location(
            "s3://my-bucket/rnaseq/batch42/processed/",
            asset_type=AssetType.folder,
            file_format="AnnData",
        )
        .with_governance(data_owner="genomics-team", is_pii=False)
        .with_experiment(sub_modality="bulk")
        .derived_from(raw_dataset_id, lineage_type=LineageType.transformed_from)
        .submit()
    )

print("Registered processed dataset:", processed_dataset_id)

## 3. List and retrieve datasets

In [ ]:
with CatalogClient(base_url=BASE_URL, api_token=API_TOKEN) as client:
    # List with filters
    page = client.datasets.list(
        project="atlas",
        modality=DatasetModality.sequencing,
        is_latest=True,
        limit=10,
    )
    print(f"{page.total} total datasets matching filter")
    for ds in page.results:
        print(f"  {ds.id}  {ds.canonical_id}  v{ds.version}  ({ds.modality})")

In [ ]:
with CatalogClient(base_url=BASE_URL, api_token=API_TOKEN) as client:
    # Get a single dataset by UUID, with lineage sideloaded
    dataset = client.datasets.get(
        processed_dataset_id,
        include_lineage=True,
    )
    print("Name:", dataset.name)
    print("Locations:", [loc.location_uri for loc in dataset.locations])
    print("Incoming edges:", dataset.incoming_lineage)
    print("Outgoing edges:", dataset.outgoing_lineage)

In [ ]:
from catalog_client import DatasetRef

with CatalogClient(base_url=BASE_URL, api_token=API_TOKEN) as client:
    # Resolve by human-readable coordinates instead of UUID
    ref = DatasetRef(
        canonical_id="pbmc-rnaseq-batch42",
        version="1.0.0",
        project="atlas",
    )
    dataset = client.datasets.get(ref)
    print("Resolved:", dataset.id, dataset.name)

## 4. Collections

In [ ]:
from catalog_client import CollectionCreate, CollectionType, CollectionUpdate

with CatalogClient(base_url=BASE_URL, api_token=API_TOKEN) as client:
    # Create
    col = client.collections.create(CollectionCreate(
        canonical_id="atlas-paper-2025",
        version="1.0.0",
        name="Atlas Paper 2025",
        collection_owner="genomics-team",
        collection_type=CollectionType.publication,
        description="Datasets used in the Atlas 2025 publication.",
    ))
    print("Created collection:", col.id)

    # Add datasets
    col = client.collections.add_dataset(col.id, raw_dataset_id)
    col = client.collections.add_dataset(col.id, processed_dataset_id)
    print("Collection updated")

    # List all collections
    page = client.collections.list(limit=5)
    print(f"{page.total} collection(s) total")

    # Rename
    col = client.collections.update(col.id, CollectionUpdate(name="Atlas Paper 2025 (final)"))
    print("Renamed to:", col.name)

## 5. Lineage edges

In [ ]:
from catalog_client import LineageEdgeCreate

with CatalogClient(base_url=BASE_URL, api_token=API_TOKEN) as client:
    # Create an edge directly (without using the registration builder)
    edge = client.lineages.create(LineageEdgeCreate(
        source_dataset_id=raw_dataset_id,
        destination_dataset_id=processed_dataset_id,
        lineage_type=LineageType.transformed_from,
    ))
    print("Edge id:", edge.id)

    # List edges originating from the raw dataset
    page = client.lineages.list(source_dataset_id=raw_dataset_id)
    print(f"{page.total} outgoing edge(s)")
    for e in page.results:
        print(f"  {e.source_dataset_id} --[{e.lineage_type.value}]--> {e.destination_dataset_id}")

## 6. Error handling

In [ ]:
from catalog_client import (
    AuthenticationError,
    CatalogConnectionError,
    CatalogServerError,
    NotFoundError,
    ValidationError,
)

with CatalogClient(base_url=BASE_URL, api_token=API_TOKEN) as client:
    try:
        client.datasets.get("00000000-0000-0000-0000-000000000000")
    except NotFoundError as e:
        print(f"404 Not Found: {e.detail}")
    except AuthenticationError:
        print("401 – invalid or expired token")
    except ValidationError as e:
        print(f"422 Validation error: {e.detail}")
    except CatalogServerError as e:
        print(f"5xx Server error ({e.status_code})")
    except CatalogConnectionError as e:
        print(f"Network error: {e}")

## 7. Async usage

`AsyncCatalogClient` has the same interface as `CatalogClient`, but every method is a coroutine.

In [ ]:
import asyncio
from catalog_client import AsyncCatalogClient

async def fetch_latest_sequencing():
    async with AsyncCatalogClient(base_url=BASE_URL, api_token=API_TOKEN) as client:
        page = await client.datasets.list(
            modality=DatasetModality.sequencing,
            is_latest=True,
            limit=5,
        )
        for ds in page.results:
            print(ds.canonical_id, ds.version)

# In a notebook, use 'await' directly if the kernel supports it:
await fetch_latest_sequencing()
# Otherwise: asyncio.run(fetch_latest_sequencing())

## Cleanup (optional)

Soft-delete the resources created above.

In [ ]:
with CatalogClient(base_url=BASE_URL, api_token=API_TOKEN) as client:
    client.collections.delete(col.id)
    client.datasets.delete(processed_dataset_id)
    client.datasets.delete(raw_dataset_id)
    print("Cleaned up.")